In [47]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [54]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline

In [55]:
#read all the words
words = open("C:/Users/Trader/Desktop/names.txt", 'r').read().splitlines()
words [:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [56]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [57]:
block_size = 3 
def build_dataset(words):
    # block size = context length aka how mnay tokesn used to predict. 
    X, Y = [],[]
    
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context) # current running context
            Y.append(ix) # prediction for the above context
            # print (''.join(itos[i] for i in context), '===>', itos[ix]) # print the context and the Y prediction
            context = context[1:] + [ix] # crop and append
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1]) #80%
Xdev,Ydev = build_dataset(words[n1:n2]) #10%
Xte, Yte = build_dataset(words[n2:]) # 10%

In [68]:
n_embd = 10
n_hidden = 200

#generator to make it reproducable. 
g = torch.Generator().manual_seed(2147483647) 
# initialize all of the relevant tensors that we will need
C = torch.randn((vocab_size,n_embd),generator = g)
W1 = torch.randn((n_embd* block_size,n_hidden), generator = g) * (5/3.2) / (n_embd*block_size) **.05
#b1 = torch.randn((n_hidden),generator = g) *.01
W2 = torch.randn((n_hidden,vocab_size),generator = g) *.01
b2 = torch.randn((vocab_size), generator = g) * 0 

bngain =  torch.ones((1,n_hidden))
bnbias = torch.zeros((1,n_hidden))
bnmean_running = torch.zeros((1,n_hidden))
bnstd_running = torch.ones((1,n_hidden))

# model parameters
parameters = [C, W1, W2, b2, bngain, bnbias]

for p in parameters:
    p.requires_grad = True
sum (p.nelement() for p in parameters)

12097

In [69]:
hpreact.mean(0,keepdim = True).shape

torch.Size([1, 200])

In [70]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range (max_steps):
    
    #batching
    ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator = g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    
    # Forward Pass
    emb = C[Xb] #embed the characters into vectors
    embcat = emb.view(emb.shape[0],-1) #concatenate the vectors. 
    hpreact = embcat  @ W1  #+ b1 # hidden layer preactivation
    #batch normalization allowing for some gain and bias 
    bnmeani = hpreact.mean(0,keepdim = True)
    bnstdi = hpreact.std(0, keepdim = True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():
        bnmean_running = .999 * bnmean_running + .001 * bnmeani
        bnstd_running = .999 * bnstd_running + .001 * bnstdi
    
    #nonlinearity layer
    h = torch.tanh(hpreact) # hidden layer
    logits = h @ W2 + b2 # output layer 
    loss = F.cross_entropy(logits,Ytr[ix]) #loss function
    #print (loss.item())
    
    #Backward Pass
    for p in parameters: 
        p.grad = None
    loss.backward()

    #learning Rate 
    lr = .1 if i <100000 else .01 #learning rate decay. 
    
    # update 
    for p in parameters:
        p.data += -lr * p.grad
    
    #track stats
    if i % 10000 == 0:
        print (f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())


      0/ 200000: 3.3239
  10000/ 200000: 2.0943
  20000/ 200000: 2.7040
  30000/ 200000: 2.1226
  40000/ 200000: 2.2850
  50000/ 200000: 1.9434
  60000/ 200000: 2.1500
  70000/ 200000: 2.3926
  80000/ 200000: 2.5929
  90000/ 200000: 2.2244
 100000/ 200000: 2.3209
 110000/ 200000: 2.4177
 120000/ 200000: 1.6638
 130000/ 200000: 2.0626
 140000/ 200000: 2.3418
 150000/ 200000: 2.0733
 160000/ 200000: 2.0318
 170000/ 200000: 2.4154
 180000/ 200000: 2.1371
 190000/ 200000: 2.1608


7

In [45]:
# calibrate the batch norm at the end of training
# no longer needes as we are doing it in the above loop as well. AKA keeping a running mean throughout training. 
with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnstd = hpreact.std(0, keepdim=True)

In [46]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  # hpreact = bngain * (hpreact - hpreact.mean(0,keepdim = True)) / hpreact.std(0, keepdim = True) + bnbias
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.125684976577759
val 2.147505760192871


In [ ]:
# Starting pointdd

In [10]:
-torch.tensor(1/27.0).log()

tensor(3.2958)

In [ ]:
# Creating the whole network in depth

In [73]:
# linear layer
class Linear:
    # initalize the network (inputs, outputs, bias) 
    def __init__(self, fan_in, fan_out, bias=True):
        # initializing the weights as a guassian distribution
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
        # initializing the biases to be 0
        self.bias = torch.zeros(fan_out) if bias else None
    
    #  W @ X + b calculation
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
          self.out += self.bias
        return self.out
        
    # returns the tensors that are the parameters 
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
  # batch normalization layer
    # 
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    # training is true to tell to use the running mean / variance vs static. 
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
   # defining the output of this layer. 
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
        # no gradient for these because there will never be a backwards pass on this. 
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

# embedding 

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

# Creating the network
C = torch.randn((vocab_size, n_embd),            generator=g)
layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]

with torch.no_grad():
  # last layer: make less confident
  layers[-1].gamma *= 0.1
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3

# parameters equals the embedding matrix plus all of the paramters in each layer. 
parameters = [C] + [p for layer in layers for p in layer.parameters()]

print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True


47024


In [74]:

#same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function
  
  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization

      0/ 200000: 3.2870
